In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from imblearn.over_sampling import ADASYN, BorderlineSMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight

# ---------------------------
# 1. Define a Cost-Sensitive Wrapper for XGBoost
# ---------------------------
class CostSensitiveXGBClassifier(XGBClassifier):
    def fit(self, X, y, **kwargs):
        # Compute balanced sample weights
        sample_weight = compute_sample_weight(class_weight='balanced', y=y)
        return super().fit(X, y, sample_weight=sample_weight, **kwargs)

# ---------------------------
# 2. Load and Split the Data
# ---------------------------
# Load features and labels.
X = pd.read_csv('X_train_cleaned.csv')
y = pd.read_csv('y_train_cleaned.csv')
y = y['label']  # assuming label column is named 'label'
# Note: In your case, labels are 0-indexed (0 to 27)

# Split data into training (80%) and validation (20%) sets with stratification.
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

num_classes = len(np.unique(y_train))
print("Number of classes:", num_classes)

# ---------------------------
# 3. Build Oversampling Pipelines with Cost-Sensitive Models
# ---------------------------
# Pipeline 1: ADASYN + Logistic Regression
pipe_lr = ImbPipeline(steps=[
    ('adasyn', ADASYN(random_state=42, n_neighbors=3)),
    ('lr', LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42))
])

# Pipeline 2: BorderlineSMOTE + Random Forest
pipe_rf = ImbPipeline(steps=[
    ('bsmote', BorderlineSMOTE(random_state=42, k_neighbors=3)),
    ('rf', RandomForestClassifier(class_weight='balanced', n_estimators=100, random_state=42))
])

# Pipeline 3: ADASYN + LightGBM
pipe_lgbm = ImbPipeline(steps=[
    ('adasyn', ADASYN(random_state=42, n_neighbors=3)),
    ('lgbm', LGBMClassifier(class_weight='balanced', n_estimators=100, random_state=42))
])

# Pipeline 4: BorderlineSMOTE + CostSensitive XGBoost
pipe_xgb = ImbPipeline(steps=[
    ('bsmote', BorderlineSMOTE(random_state=42, k_neighbors=3)),
    ('xgb', CostSensitiveXGBClassifier(
                n_estimators=100,
                objective='multi:softprob',
                num_class=num_classes,
                use_label_encoder=False,
                eval_metric='mlogloss',
                random_state=42))
])

# ---------------------------
# 4. Ensemble Voting: Combine the Pipelines
# ---------------------------
ensemble = VotingClassifier(
    estimators=[
        ('lr', pipe_lr),
        ('rf', pipe_rf),
        ('lgbm', pipe_lgbm),
        ('xgb', pipe_xgb)
    ],
    voting='soft'  # soft voting, uses predicted probabilities
)

# ---------------------------
# 5. Train the Ensemble Model
# ---------------------------
ensemble.fit(X_train, y_train)

# ---------------------------
# 6. Make Predictions and Evaluate
# ---------------------------
y_pred = ensemble.predict(X_val)
acc = accuracy_score(y_val, y_pred)
print("Ensemble Validation Accuracy:", acc)
print("Ensemble Classification Report:")
print(classification_report(y_val, y_pred))


Number of classes: 28
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.075982 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 76500
[LightGBM] [Info] Number of data points in the train set: 98038, number of used features: 300
[LightGBM] [Info] Start training from score -3.332205
[LightGBM] [Info] Start training from score -3.332205
[LightGBM] [Info] Start training from score -3.332205
[LightGBM] [Info] Start training from score -3.332204
[LightGBM] [Info] Start training from score -3.332205
[LightGBM] [Info] Start training from score -3.332205
[LightGBM] [Info] Start training from score -3.332205
[LightGBM] [Info] Start training from score -3.332204
[LightGBM] [Info] Start training from score -3.332204
[LightGBM] [Info] Start training from score -3.332204
[LightGBM] [Info] Start training from score -3.332205
[LightGBM] [Info] Start training from score -3.332205
[LightGBM] [Info] Start training from s

/Users/davinagreen/anaconda3/lib/python3.11/site-packages/xgboost/core.py:158: UserWarning: [13:28:54] WARNING: /var/folders/k1/30mswbxs7r1g6zwn8y4fyt500000gp/T/abs_d9k8pmaj4_/croot/xgboost-split_1724073758172/work/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Ensemble Validation Accuracy: 0.7853457172342622
Ensemble Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         3
           1       0.00      0.00      0.00         1
           2       0.00      0.00      0.00         2
           3       0.57      0.31      0.40        13
           4       0.60      0.70      0.65        46
           5       0.91      0.97      0.94       875
           6       0.91      0.95      0.93       106
           7       0.62      0.38      0.47        21
           8       0.76      0.79      0.77        98
           9       0.00      0.00      0.00         5
          10       0.70      0.85      0.77       208
          11       0.64      0.75      0.69        12
          12       0.47      0.52      0.50        88
          13       0.00      0.00      0.00        12
          14       0.23      0.06      0.09        52
          15       0.80      0.80      0.80         5


/Users/davinagreen/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/davinagreen/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/davinagreen/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [2]:
# ------------------------------------------------------------------
# 0.  If the ensemble is still in memory just save it  -------------
# ------------------------------------------------------------------
import joblib   # part of the scikit‑learn stack
joblib.dump(ensemble, "vote_ensemble.pkl")      # ≈ a few MB on disk
print("Model saved to vote_ensemble.pkl")

# ------------------------------------------------------------------
# 1.  Later / in another notebook: reload the model ----------------
# ------------------------------------------------------------------
import joblib, pandas as pd
ensemble = joblib.load("vote_ensemble.pkl")

# ------------------------------------------------------------------
# 2.  Load the *unseen* test features ------------------------------
# ------------------------------------------------------------------
X_test = pd.read_csv("X_test_1.csv")

# ⚠️  Make sure the column order and preprocessing match training
#     (e.g., same dummies, same scaling).  If you trained directly on
#     the cleaned CSV without extra transforms, loading the file is
#     enough.  Otherwise apply the same preprocessing pipeline here.

# ------------------------------------------------------------------
# 3.  Predict labels and/or probabilities --------------------------
# ------------------------------------------------------------------
y_pred       = ensemble.predict(X_test)
y_proba      = ensemble.predict_proba(X_test)   # shape = (n_samples, 28)

# ------------------------------------------------------------------
# 4.  Export predictions to disk -----------------------------------
# ------------------------------------------------------------------
out = pd.DataFrame({
        "id"   : X_test.index,     # or any identifier column you have
        "pred" : y_pred
})
out.to_csv("test_predictions.csv", index=False)
print("Wrote test_predictions.csv")

# Optional: if you also need class‑probabilities
proba_df = pd.DataFrame(y_proba, columns=[f"class_{c}"
                                          for c in ensemble.classes_])
proba_df.insert(0, "id", X_test.index)
proba_df.to_csv("test_pred_proba.csv", index=False)
print("Wrote test_pred_proba.csv")



Model saved to vote_ensemble.pkl
Wrote test_predictions.csv
Wrote test_pred_proba.csv


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import classification_report, accuracy_score, log_loss
from imblearn.over_sampling import ADASYN, BorderlineSMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight

# ---------------------------
# 1. Define a Cost-Sensitive Wrapper for XGBoost
# ---------------------------
class CostSensitiveXGBClassifier(XGBClassifier):
    def fit(self, X, y, **kwargs):
        # Compute balanced sample weights.
        sample_weight = compute_sample_weight(class_weight='balanced', y=y)
        return super().fit(X, y, sample_weight=sample_weight, **kwargs)

# ---------------------------
# 2. Define a function to tune logistic regression via GridSearchCV
# ---------------------------
def lr_train(X, y, cw=None):
    """
    Tune logistic regression using a grid search over parameter C.
    Returns the best logistic regression estimator.
    """
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    C_values = np.logspace(-2, 2, 10)
    lr_param_grid = {"C": C_values}
    
    lr_model = LogisticRegression(random_state=42, max_iter=1000, class_weight=cw)
    lr_grid = GridSearchCV(lr_model, param_grid=lr_param_grid, cv=kf,
                           scoring='accuracy', n_jobs=4, verbose=1)
    lr_grid.fit(X, y)
    best_lr = lr_grid.best_estimator_
    print(f"\nBest CV accuracy : {lr_grid.best_score_:.4f}")
    print("Best C           :", lr_grid.best_params_["C"])
    return best_lr

# ---------------------------
# 3. Load and Split the Data
# ---------------------------
X = pd.read_csv('X_train_cleaned.csv')
y = pd.read_csv('y_train_cleaned.csv')
y = y['label']   # assuming label column is named 'label'
# Note: Labels are assumed to be 0-indexed (0 to 27)

# Split data into training (80%) and validation (20%) sets with stratification.
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

num_classes = len(np.unique(y_train))
print("Number of classes:", num_classes)

# ---------------------------
# 4. Build Oversampling Pipelines for RF, LGBM, and XGBoost
# ---------------------------
# Pipeline for Random Forest: using BorderlineSMOTE
pipe_rf = ImbPipeline(steps=[
    ('bsmote', BorderlineSMOTE(random_state=42, k_neighbors=3)),
    ('rf', RandomForestClassifier(class_weight='balanced', n_estimators=100, random_state=42))
])

# Pipeline for LightGBM: using ADASYN
pipe_lgbm = ImbPipeline(steps=[
    ('adasyn', ADASYN(random_state=42, n_neighbors=3)),
    ('lgbm', LGBMClassifier(class_weight='balanced', n_estimators=100, random_state=42))
])

# Pipeline for XGBoost: using BorderlineSMOTE and our cost-sensitive wrapper
pipe_xgb = ImbPipeline(steps=[
    ('bsmote', BorderlineSMOTE(random_state=42, k_neighbors=3)),
    ('xgb', CostSensitiveXGBClassifier(
                n_estimators=100,
                objective='multi:softprob',
                num_class=num_classes,
                use_label_encoder=False,
                eval_metric='mlogloss',
                random_state=42))
])

# ---------------------------
# 5. Get the Best Logistic Regression Model via GridSearchCV
# ---------------------------
# Note: Here we do *not* put logistic regression inside an oversampling pipeline.
best_lr = lr_train(X_train, y_train, cw='balanced')

# ---------------------------
# 6. Ensemble Voting: Combine the Models
# ---------------------------
ensemble = VotingClassifier(
    estimators=[
        ('lr_best', best_lr),   # the tuned logistic regression model
        ('rf', pipe_rf),
        ('lgbm', pipe_lgbm),
        ('xgb', pipe_xgb)
    ],
    voting='soft'
    # soft voting: average predicted probabilities
)

# ---------------------------
# 7. Train the Ensemble Model
# ---------------------------
ensemble.fit(X_train, y_train)

# ---------------------------
# 8. Make Predictions and Evaluate on the Validation Set
# ---------------------------
y_pred = ensemble.predict(X_val)
acc = accuracy_score(y_val, y_pred)
print("Ensemble Validation Accuracy:", acc)
print("Ensemble Classification Report:")
print(classification_report(y_val, y_pred))

# ---------------------------
# 9. Define Function to Compute Weighted Log Loss
# ---------------------------
def weighted_log_loss_from_labels(y_true, y_pred_proba, n_classes):
    """
    Compute weighted log loss where each sample's loss is weighted by
    the inverse of its class frequency (normalized to sum to 1).
    """
    # Convert true labels to one-hot encoding
    y_true_ohe = (np.arange(n_classes) == np.array(y_true)[:, None]).astype(int)
    # Calculate class frequencies and determine weights: w_c = 1 / frequency
    class_counts = np.sum(y_true_ohe, axis=0)
    class_weights = 1.0 / class_counts
    # Normalize weights so they sum to 1
    class_weights /= np.sum(class_weights)
    # Get sample weights based on the true label
    sample_weights = np.sum(y_true_ohe * class_weights, axis=1)
    # Compute per-sample log loss (add epsilon to avoid log(0))
    eps = 1e-15
    log_losses = -np.sum(y_true_ohe * np.log(y_pred_proba + eps), axis=1)
    return np.mean(sample_weights * log_losses)

# ---------------------------
# 10. Compute and Print the Weighted Average Loss on the Validation Set
# ---------------------------
y_val_proba = ensemble.predict_proba(X_val)
weighted_loss = weighted_log_loss_from_labels(y_val, y_val_proba, num_classes)
print("Weighted Average Loss (Validation):", weighted_loss)


Number of classes: 28
Fitting 5 folds for each of 10 candidates, totalling 50 fits


/Users/davinagreen/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/davinagreen/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html


Best CV accuracy : 0.6784
Best C           : 0.5994842503189409
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.071232 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 76500
[LightGBM] [Info] Number of data points in the train set: 98038, number of used features: 300
[LightGBM] [Info] Start training from score -3.332205
[LightGBM] [Info] Start training from score -3.332205
[LightGBM] [Info] Start training from score -3.332205
[LightGBM] [Info] Start training from score -3.332204
[LightGBM] [Info] Start training from score -3.332205
[LightGBM] [Info] Start training from score -3.332205
[LightGBM] [Info] Start training from score -3.332205
[LightGBM] [Info] Start training from score -3.332204
[LightGBM] [Info] Start training from score -3.332204
[LightGBM] [Info] Start training from score -3.332204
[LightGBM] [Info] Start training from score -3.332205
[LightGBM] [Info] Start training from score -3.332

/Users/davinagreen/anaconda3/lib/python3.11/site-packages/xgboost/core.py:158: UserWarning: [21:02:52] WARNING: /var/folders/k1/30mswbxs7r1g6zwn8y4fyt500000gp/T/abs_d9k8pmaj4_/croot/xgboost-split_1724073758172/work/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Ensemble Validation Accuracy: 0.7868937048503611
Ensemble Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         3
           1       0.00      0.00      0.00         1
           2       0.00      0.00      0.00         2
           3       0.62      0.38      0.48        13
           4       0.59      0.70      0.64        46
           5       0.92      0.97      0.94       875
           6       0.90      0.95      0.93       106
           7       0.62      0.38      0.47        21
           8       0.75      0.77      0.76        98
           9       0.00      0.00      0.00         5
          10       0.70      0.86      0.77       208
          11       0.60      0.75      0.67        12
          12       0.48      0.53      0.51        88
          13       0.00      0.00      0.00        12
          14       0.21      0.06      0.09        52
          15       0.80      0.80      0.80         5


/Users/davinagreen/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/davinagreen/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/davinagreen/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Weighted Average Loss (Validation): 0.007399321646342293


In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import classification_report, accuracy_score
from imblearn.over_sampling import ADASYN, BorderlineSMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.ensemble import VotingClassifier

# 1. Cost-Sensitive Wrapper for XGBoost
class CostSensitiveXGBClassifier(XGBClassifier):
    def fit(self, X, y, **kwargs):
        sample_weight = compute_sample_weight(class_weight='balanced', y=y)
        return super().fit(X, y, sample_weight=sample_weight, **kwargs)

# 2. Logistic Regression Tuning Function
def lr_train(X, y, cw=None):
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    C_values = np.logspace(-2, 2, 10)
    grid = GridSearchCV(
        LogisticRegression(random_state=42, max_iter=1000, class_weight=cw),
        {'C': C_values},
        cv=kf,
        scoring='accuracy',
        n_jobs=4,
        verbose=1
    )
    grid.fit(X, y)
    print(f"\nBest LR CV accuracy: {grid.best_score_:.4f}")
    print("Best C:", grid.best_params_['C'])
    return grid.best_estimator_

# 3. Load & Split Data
X = pd.read_csv('X_train_cleaned.csv')
y = pd.read_csv('y_train_cleaned.csv')['label']
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
num_classes = len(np.unique(y_train))
print("Number of classes:", num_classes)

# 4. Define Oversampling + Model Pipelines

# 4a. ADASYN + Logistic Regression
pipe_lr = ImbPipeline(steps=[
    ('adasyn', ADASYN(random_state=42, n_neighbors=3)),
    ('lr', LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42))
])

# 4b. BorderlineSMOTE + SVM
pipe_svm = ImbPipeline(steps=[
    ('bsmote', BorderlineSMOTE(random_state=42, k_neighbors=3)),
    ('svm', SVC(class_weight='balanced', probability=True, random_state=42))
])

from lightgbm import LGBMClassifier, early_stopping

# 1. 先做过采样
X_tr_ada, y_tr_ada = ADASYN(random_state=42, n_neighbors=3) \
                       .fit_resample(X_train, y_train)

# 2. 构造带 callback 的 LGBMClassifier
lgbm_tuner = LGBMClassifier(
    learning_rate=0.05,
    n_estimators=500,            # 上限
    class_weight='balanced',
    random_state=42,
    callbacks=[early_stopping(stopping_rounds=50)]
)

# 3. 只传 eval_set（不要传 early_stopping_rounds）
lgbm_tuner.fit(
    X_tr_ada,
    y_tr_ada,
    eval_set=[(X_val, y_val)],
    verbose=False
)

# 4. 拿到最佳迭代次数
best_iter_lgbm = lgbm_tuner.best_iteration_
print("LGBM 最佳迭代次数：", best_iter_lgbm)


# 4d. BorderlineSMOTE + Cost-Sensitive XGBoost
pipe_xgb = ImbPipeline(steps=[
    ('bsmote', BorderlineSMOTE(random_state=42, k_neighbors=3)),
    ('xgb', CostSensitiveXGBClassifier(
        n_estimators=100,
        objective='multi:softprob',
        num_class=num_classes,
        use_label_encoder=False,
        eval_metric='mlogloss',
        random_state=42
    ))
])

# 5. Tune Logistic Regression (not in pipeline)
best_lr = lr_train(X_train, y_train, cw='balanced')

# 6. Ensemble Voting (replace RF with SVM)
ensemble = VotingClassifier(
    estimators=[
        ('lr_best', best_lr),
        ('svm', pipe_svm),
        ('lgbm', pipe_lgbm),
        ('xgb', pipe_xgb)
    ],
    voting='soft'
)

# 7. Train Ensemble
ensemble.fit(X_train, y_train)

# 8. Evaluate on Validation Set
y_pred = ensemble.predict(X_val)
print("Ensemble Validation Accuracy:", accuracy_score(y_val, y_pred))
print("Ensemble Classification Report:")
print(classification_report(y_val, y_pred, zero_division=0))


Number of classes: 28
Fitting 5 folds for each of 10 candidates, totalling 50 fits


/Users/davinagreen/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/davinagreen/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html


Best LR CV accuracy: 0.6784
Best C: 0.5994842503189409
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.118821 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 76500
[LightGBM] [Info] Number of data points in the train set: 98038, number of used features: 300
[LightGBM] [Info] Start training from score -3.332205
[LightGBM] [Info] Start training from score -3.332205
[LightGBM] [Info] Start training from score -3.332205
[LightGBM] [Info] Start training from score -3.332204
[LightGBM] [Info] Start training from score -3.332205
[LightGBM] [Info] Start training from score -3.332205
[LightGBM] [Info] Start training from score -3.332205
[LightGBM] [Info] Start training from score -3.332204
[LightGBM] [Info] Start training from score -3.332204
[LightGBM] [Info] Start training from score -3.332204
[LightGBM] [Info] Start training from score -3.332205
[LightGBM] [Info] Start training from score -3.332205
[Ligh

/Users/davinagreen/anaconda3/lib/python3.11/site-packages/xgboost/core.py:158: UserWarning: [00:14:07] WARNING: /var/folders/k1/30mswbxs7r1g6zwn8y4fyt500000gp/T/abs_d9k8pmaj4_/croot/xgboost-split_1724073758172/work/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Ensemble Validation Accuracy: 0.7915376676986584
Ensemble Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         3
           1       0.00      0.00      0.00         1
           2       0.00      0.00      0.00         2
           3       0.62      0.38      0.48        13
           4       0.57      0.70      0.63        46
           5       0.91      0.97      0.94       875
           6       0.89      0.95      0.92       106
           7       0.67      0.38      0.48        21
           8       0.77      0.77      0.77        98
           9       0.00      0.00      0.00         5
          10       0.70      0.86      0.77       208
          11       0.60      0.75      0.67        12
          12       0.51      0.53      0.52        88
          13       0.00      0.00      0.00        12
          14       0.17      0.04      0.06        52
          15       0.80      0.80      0.80         5


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import classification_report, accuracy_score
from imblearn.over_sampling import ADASYN, BorderlineSMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.ensemble import VotingClassifier

# 1. Cost‑Sensitive Wrapper for XGBoost
class CostSensitiveXGBClassifier(XGBClassifier):
    def fit(self, X, y, **kwargs):
        sample_weight = compute_sample_weight(class_weight='balanced', y=y)
        return super().fit(X, y, sample_weight=sample_weight, **kwargs)



# 3. Load & Split Data
X = pd.read_csv('X_train_cleaned.csv')
y = pd.read_csv('y_train_cleaned.csv')['label']
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
num_classes = len(np.unique(y_train))

# 4. Define Oversampling + Model Pipelines with Best Params

# 4a. ADASYN + Logistic Regression (C=0.215, lbfgs, max_iter=2000)
pipe_lr = ImbPipeline(steps=[
    ('adasyn', ADASYN(random_state=42, n_neighbors=3)),
    ('lr', LogisticRegression(
        C=0.215,
        solver='lbfgs',
        class_weight='balanced',
        max_iter=2000,
        random_state=42
    ))
])

# 4b. BorderlineSMOTE + SVM (C=100, kernel=rbf, gamma=auto)
pipe_svm = ImbPipeline(steps=[
    ('bsmote', BorderlineSMOTE(random_state=42, k_neighbors=3)),
    ('svm', SVC(
        C=100,
        kernel='rbf',
        gamma='auto',
        class_weight='balanced',
        probability=True,
        random_state=42
    ))
])

# 4c. ADASYN + LightGBM (num_leaves=80, max_depth=7, lr=0.05, n_estimators=300, feature_fraction=0.7)
pipe_lgbm = ImbPipeline(steps=[
    ('adasyn', ADASYN(random_state=42, n_neighbors=3)),
    ('lgbm', LGBMClassifier(
        class_weight='balanced',
        num_leaves=80,
        max_depth=7,
        learning_rate=0.05,
        n_estimators=300,
        feature_fraction=0.7,
        random_state=42
    ))
])

# 4d. BorderlineSMOTE + Cost‑Sensitive XGBoost (max_depth=5, lr=0.1, n_estimators=200, subsample=0.8, colsample_bytree=0.8)
pipe_xgb = ImbPipeline(steps=[
    ('bsmote', BorderlineSMOTE(random_state=42, k_neighbors=3)),
    ('xgb', CostSensitiveXGBClassifier(
        max_depth=5,
        learning_rate=0.1,
        n_estimators=200,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='multi:softprob',
        num_class=num_classes,
        eval_metric='mlogloss',
        random_state=42
    ))
])



# 6. Build Ensemble with tuned branches
ensemble = VotingClassifier(
    estimators=[
        ('lr', pipe_lr),
        ('svm', pipe_svm),
        ('lgbm', pipe_lgbm),
        ('xgb', pipe_xgb)
    ],
    voting='soft'
)

# 7. Train Ensemble
ensemble.fit(X_train, y_train)

# 8. Evaluate on Validation Set
y_pred = ensemble.predict(X_val)
print("Ensemble Validation Accuracy:", accuracy_score(y_val, y_pred))
print("Ensemble Classification Report:")
print(classification_report(y_val, y_pred, zero_division=0))


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import classification_report, accuracy_score
from imblearn.over_sampling import ADASYN, BorderlineSMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.ensemble import VotingClassifier
from lightgbm.callback import early_stopping
from xgboost.callback import EarlyStopping



# 1. Cost-Sensitive Wrapper for XGBoost
class CostSensitiveXGBClassifier(XGBClassifier):
    def fit(self, X, y, **kwargs):
        sample_weight = compute_sample_weight(class_weight='balanced', y=y)
        return super().fit(X, y, sample_weight=sample_weight, **kwargs)



# 2. Load & Split Data
X = pd.read_csv('X_train_cleaned.csv')
y = pd.read_csv('y_train_cleaned.csv')['label']
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
num_classes = len(np.unique(y_train))
print("Number of classes:", num_classes)

# 3. Define Oversampling + Model Pipelines

# 3a. ADASYN + Logistic Regression
pipe_lr = ImbPipeline(steps=[
    ('adasyn', ADASYN(random_state=42, n_neighbors=3)),
    ('lr', LogisticRegression(C=0.215,solver='lbfgs', class_weight='balanced', max_iter=2000, random_state=42))
])

# 3b. BorderlineSMOTE + SVM
pipe_svm = ImbPipeline(steps=[
    ('bsmote', BorderlineSMOTE(random_state=42, k_neighbors=3)),
    ('svm', SVC(C=100,gamma="auto", class_weight='balanced', probability=True, random_state=42))
])

# 3c. ADASYN + LightGBM
pipe_lgbm = ImbPipeline(steps=[
    ('adasyn', ADASYN(random_state=42, n_neighbors=3)),
    ('lgbm', LGBMClassifier(learning_rate=0.05,
                            n_estimators=300,
                            num_leaves=80,
                            max_depth=7,
                            feature_fraction=0.7,
                            class_weight='balanced',
                            random_state=42))
])


# 3d. BorderlineSMOTE + Cost-Sensitive XGBoost
pipe_xgb = ImbPipeline(steps=[
    ('bsmote', BorderlineSMOTE(random_state=42, k_neighbors=3)),
    ('xgb', CostSensitiveXGBClassifier(
        learning_rate=0.1,
        n_estimators=200,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='multi:softprob',
        num_class=num_classes,
        eval_metric='mlogloss',
        random_state=42
    ))
])



# 5. Ensemble Voting (replace RF with SVM)
ensemble = VotingClassifier(
    estimators=[
        ('lr_best',pipe_lr),
        ('svm', pipe_svm),
        ('lgbm', pipe_lgbm),
        ('xgb', pipe_xgb)
    ],
    voting='soft'
)

# 6. Train Ensemble
ensemble.fit(X_train, y_train)

# 7. Evaluate on Validation Set
y_pred = ensemble.predict(X_val)
print("Ensemble Validation Accuracy:", accuracy_score(y_val, y_pred))
print("Ensemble Classification Report:")
print(classification_report(y_val, y_pred, zero_division=0))


Number of classes: 28
[LightGBM] [Warning] feature_fraction is set=0.7, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.7
[LightGBM] [Warning] feature_fraction is set=0.7, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.7
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.064698 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 76500
[LightGBM] [Info] Number of data points in the train set: 98038, number of used features: 300
[LightGBM] [Info] Start training from score -3.332205
[LightGBM] [Info] Start training from score -3.332205
[LightGBM] [Info] Start training from score -3.332205
[LightGBM] [Info] Start training from score -3.332204
[LightGBM] [Info] Start training from score -3.332205
[LightGBM] [Info] Start training from score -3.332205
[LightGBM] [Info] Start training from score -3.332205
[LightGBM] [Info] Start training from score -3.332204
[LightG